In [1]:
import os
import zipfile
import random
import shutil
from google.colab import drive

In [2]:
# --- 1. Mount Google Drive ---
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# --- 2. Define Paths ---
#  *Update this path to the actual location of your zip file in Google Drive*
zip_file_path = '/content/drive/MyDrive/Tea_Leaf_Disease.zip'  # Example path. CHANGE THIS!
extracted_folder_path = '/content/extracted_tea_data'  # Temporary folder for extraction

In [4]:
# --- Paths within the final dataset directory ---
final_dataset_dir = '/content/Tea_Leaf_Final_Dataset'
train_dir = os.path.join(final_dataset_dir, 'train')
test_dir = os.path.join(final_dataset_dir, 'test')
val_dir = os.path.join(final_dataset_dir, 'valid')
base_dir = '/content/Tea_Leaf_Disease' #where extracted data will be moved

In [5]:
# --- Path for the final zipped dataset on Google Drive ---
# *Update this to where you want to SAVE the zipped dataset*
output_zip_path = '/content/drive/MyDrive/Tea_Leaf_Final_Dataset.zip'  # Example. CHANGE THIS!

In [6]:
# --- 3. Unzip the Dataset ---
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_folder_path)

In [7]:
# --- 3.5 Move to /content ---
# Move the contents from the extracted folder to /content
extracted_main_folder = os.path.join(extracted_folder_path, 'Tea_Leaf_Disease')  # Adjust if necessary
if os.path.exists(extracted_main_folder):
  for item in os.listdir(extracted_main_folder):
    s = os.path.join(extracted_main_folder, item)
    d = os.path.join(base_dir, item)
    if os.path.isdir(s):
      shutil.move(s, d) #move extracted folder to /content
    else:
      shutil.copy2(s, d) #it can files too, copy those
else:
    print(f"Error:  Expected folder '{extracted_main_folder}' not found.")
    exit()

In [8]:
# --- 4. Create the main "Tea_Leaf_Final_Dataset" and its subdirectories ---
os.makedirs(final_dataset_dir, exist_ok=True)  # Create the main directory
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

In [9]:
# --- 5. Get Class Names ---
class_names = [folder_name for folder_name in os.listdir(base_dir)
               if os.path.isdir(os.path.join(base_dir, folder_name))]
print(f"Found classes: {class_names}")

Found classes: ['helopeltis', 'brown_blight', 'healthy', 'gray_blight', 'algal_spot', 'red_spot']


In [10]:
# --- 6. Split Data (Train/Test/Validation) ---
train_ratio = 0.7
test_ratio = 0.15
val_ratio = 0.15

In [11]:
for class_name in class_names:
    # Create class-specific subdirectories in train, test, and val.
    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

    # Get all image paths.
    class_dir = os.path.join(base_dir, class_name)
    image_paths = [os.path.join(class_dir, img) for img in os.listdir(class_dir)
                   if os.path.isfile(os.path.join(class_dir, img))]

    # Shuffle the images.
    random.shuffle(image_paths)

    # Calculate split indices.
    num_images = len(image_paths)
    train_split = int(num_images * train_ratio)
    test_split = int(num_images * (train_ratio + test_ratio))

    # Split the images.
    train_images = image_paths[:train_split]
    test_images = image_paths[train_split:test_split]
    val_images = image_paths[test_split:]

    # Copy images to their respective directories.
    for img_path in train_images:
        shutil.copy(img_path, os.path.join(train_dir, class_name))
    for img_path in test_images:
        shutil.copy(img_path, os.path.join(test_dir, class_name))
    for img_path in val_images:
        shutil.copy(img_path, os.path.join(val_dir, class_name))
    print(f"Class '{class_name}' split: Train={len(train_images)}, Test={len(test_images)}, Val={len(val_images)}")

Class 'helopeltis' split: Train=700, Test=150, Val=150
Class 'brown_blight' split: Train=606, Test=130, Val=131
Class 'healthy' split: Train=700, Test=150, Val=150
Class 'gray_blight' split: Train=700, Test=150, Val=150
Class 'algal_spot' split: Train=700, Test=150, Val=150
Class 'red_spot' split: Train=700, Test=150, Val=150


In [12]:
# --- 7.  (Optional) Cleanup Extracted ---
# Remove the temporary extracted folder.
shutil.rmtree(extracted_folder_path)
shutil.rmtree(base_dir)

In [13]:
# --- 8. Zip the Final Dataset and Save to Google Drive ---
def zipdir(path, ziph):
    # ziph is zipfile handle
    for root, dirs, files in os.walk(path):
        for file in files:
            # Create a relative path for files within the zip
            rel_path = os.path.relpath(os.path.join(root, file), os.path.join(path, '..'))
            ziph.write(os.path.join(root, file), rel_path)

In [14]:
with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipdir(final_dataset_dir, zipf)

In [15]:
# --- 9. (Optional) Clean up unzipped folder ---
shutil.rmtree(final_dataset_dir)

In [16]:
print(f"Dataset organization and zipping complete!  Zipped file saved to: {output_zip_path}")

Dataset organization and zipping complete!  Zipped file saved to: /content/drive/MyDrive/Tea_Leaf_Final_Dataset.zip
